# Regression without Sirakaya's estimates

This notebook re-estimates the proportional-hazards model directly from the imputed covariates. 
- It does not uses any coefficient of Sirakaya's paper anymore

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lifelines import CoxPHFitter
from scipy.stats import chi2

import input

RESULTS_DIR = Path("results/regression-without-sirakaya")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR

## Load covariates and recompute county statistics

Use the imputed covariate table and recompute the two county statistics needed by the Cox formula.

In [ ]:
_, df, _, _ = input.load_data()

county_statistics = (
    df.groupby("code_county")
    .agg(
        mean_re_time=("time", "mean"),
        percent_re=("observed", "mean"),
    )
)

print("imputed rows:", len(df))
print("counties:", len(county_statistics))
display(county_statistics.head())

## Assemble the modeling frame

Each categorical variable is rounded after imputation, restricted to its documented level range, and marked as categorical. Level 1 is the reference category.

In [ ]:
categorical_levels = {
    "employ": 3,
    "felony_prior_conviction": 3,
    "offense_type": 8,
    "sex": 2,
    "ethnicity": 2,
    "drug_abuse": 3,
    "race": 5,
    "supervision": 6,
    "age_dist": 6,
}
county_columns = ["house_female", "tax_property"]
county_statistic_columns = ["mean_re_time", "percent_re"]

cox_data = df.merge(
    county_statistics,
    left_on="code_county",
    right_index=True,
    how="left",
)

model_columns = (
    list(categorical_levels)
    + county_columns
    + county_statistic_columns
    + ["felony_arrest", "time", "observed"]
)
cox_data = cox_data[model_columns].dropna().copy()
# Felony arrests during the observed probation period.
cox_data["offenses"] = cox_data["felony_arrest"].round()
cox_data["time"] = cox_data["time"].clip(lower=1.0)
cox_data["observed"] = (cox_data["observed"] > 0).astype(int)

for column, number_of_levels in categorical_levels.items():
    cox_data[column] = (
        cox_data[column]
        .round()
        .clip(1, number_of_levels)
        .astype(int)
        .astype("category")
    )

print("modeling frame:", cox_data.shape)
print("observed events:", int(cox_data["observed"].sum()))
display(cox_data.head())

## Fit the Cox proportional-hazards model

This is an unpenalized maximum-likelihood fit. If it reports convergence or singular-matrix problems, inspect collinearity, sparse levels, and event counts before changing the specification.

In [ ]:
cox_formula = " + ".join(
    list(categorical_levels)
    + county_columns
    + county_statistic_columns
    + ["offenses"]
)

cox_model = CoxPHFitter(penalizer=0.0)
cox_model.fit(
    cox_data,
    duration_col="time",
    event_col="observed",
    formula=cox_formula,
    show_progress=True,
)

cox_model.print_summary(decimals=4)
cox_model.summary.to_csv(
    RESULTS_DIR / "cox_coefficient_summary.csv"
)

## Joint tests for categorical variables

The coefficient table tests individual dummy levels. This cell adds one joint Wald test for each complete categorical variable.

In [ ]:
coefficients = cox_model.params_
covariance = cox_model.variance_matrix_
wald_rows = []

for column in categorical_levels:
    coefficient_names = [
        name
        for name in coefficients.index
        if name == column
        or name.startswith(column + "[")
        or name.startswith(column + "_")
    ]

    beta = coefficients.loc[coefficient_names].to_numpy()
    covariance_block = covariance.loc[
        coefficient_names, coefficient_names
    ].to_numpy()
    statistic = float(
        beta @ np.linalg.solve(covariance_block, beta)
    )

    wald_rows.append(
        {
            "covariate": column,
            "df": len(coefficient_names),
            "wald_chi2": statistic,
            "p_value": chi2.sf(statistic, len(coefficient_names)),
        }
    )

joint_wald_tests = pd.DataFrame(wald_rows)
joint_wald_tests.to_csv(
    RESULTS_DIR / "cox_joint_wald_tests.csv",
    index=False,
)
display(joint_wald_tests)

## Save scores, baseline survival, and plots

In [ ]:
cox_data["score_fitted"] = cox_model.predict_log_partial_hazard(
    cox_data
)
baseline_survival = cox_model.baseline_survival_

cox_data.to_csv(
    RESULTS_DIR / "cox_individual_scores.csv",
    index=False,
)
baseline_survival.to_csv(
    RESULTS_DIR / "cox_baseline_survival.csv"
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(
    baseline_survival.index,
    baseline_survival.iloc[:, 0],
    linewidth=2,
)
ax.set_xlabel("Time (days)")
ax.set_ylabel("Baseline survival")
ax.set_title("Baseline survival without Sirakaya estimates")
ax.set_ylim(0, 1.02)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(
    RESULTS_DIR / "cox_baseline_survival.png",
    dpi=300,
)
plt.show()

cox_model.plot_partial_effects_on_outcome(
    covariates="offenses",
    values=[0, 2, 6, 10],
)
plt.tight_layout()
plt.savefig(
    RESULTS_DIR / "cox_offense_partial_effects.png",
    dpi=300,
)
plt.show()

## Optional proportional-hazards diagnostics

The diagnostic report can be long, so run it separately when needed.

In [ ]:
cox_model.check_assumptions(cox_data)

## Handoff

The files under `results/regression-without-sirakaya/` are analysis outputs; `run_policy.py` does not load them automatically. If this fit supports a model change, update and record the simulator code before running the policy simulation.